# 2.0 Bloom Classifier – Baseline B0 + B1

**Tác giả:** Đăng (TV1 – Tầng mô hình)  
**Ngày:** 2026-10-30  
**Mục tiêu:**
- B0: Rule-based baseline (danh sách động từ Bloom)
- B1: TF-IDF + Logistic Regression, 5-fold CV theo môn
- Lưu mô hình tốt nhất vào `models/bloom_model.pkl`
- Ghi kết quả vào `results/metrics.json`

In [ ]:
import json
import pickle
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
import scipy.sparse as sp

ROOT = Path('..').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
with open(ROOT / 'config' / 'params.yaml') as f:
    CFG = yaml.safe_load(f)

RANDOM_SEED = CFG.get('random_seed', 42)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

FIG_DIR = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = ROOT / 'results' / 'metrics.json'

BLOOM_NAMES = ['Ngoài miền','Nhớ','Hiểu','Áp dụng','Phân tích','Đánh giá','Sáng tạo']
print('Imports OK')

## 1. Nạp dữ liệu CLO đã gán nhãn

In [ ]:
CLO_CSV = ROOT / 'data' / 'processed' / 'clos_labeled.csv'

if CLO_CSV.exists():
    df = pd.read_csv(CLO_CSV)
    print(f'Nạp từ file: {len(df)} CLO')
else:
    print('Không có clos_labeled.csv → dùng 28 CLO tổng hợp')
    SYNTHETIC_CLOS = [
        ('841020', 'Trình bày định nghĩa biến và kiểu dữ liệu', 1),
        ('841020', 'Liệt kê các toán tử cơ bản trong Python', 1),
        ('841020', 'Giải thích cách hoạt động của vòng lặp for', 2),
        ('841020', 'Mô tả sự khác biệt giữa list và tuple', 2),
        ('841020', 'Viết chương trình tính giai thừa bằng đệ quy', 3),
        ('841020', 'Áp dụng hàm lambda để xử lý danh sách', 3),
        ('841303', 'Nêu định nghĩa cấu trúc stack và queue', 1),
        ('841303', 'Giải thích độ phức tạp thuật toán O(n log n)', 2),
        ('841303', 'Cài đặt cây nhị phân tìm kiếm bằng Java', 3),
        ('841303', 'Phân tích ưu nhược điểm của các thuật toán sắp xếp', 4),
        ('841303', 'Lựa chọn cấu trúc dữ liệu phù hợp cho bài toán cụ thể', 5),
        ('841109', 'Kể tên các thành phần cơ bản của mạng máy tính', 1),
        ('841109', 'Giải thích mô hình OSI 7 tầng', 2),
        ('841109', 'Cấu hình router và switch cơ bản', 3),
        ('841109', 'So sánh TCP và UDP về độ tin cậy và hiệu năng', 4),
        ('841109', 'Thiết kế mạng LAN cho văn phòng 50 người', 6),
        ('841434', 'Định nghĩa khái niệm transaction trong CSDL', 1),
        ('841434', 'Mô tả mô hình quan hệ và các ràng buộc toàn vẹn', 2),
        ('841434', 'Viết câu truy vấn SQL JOIN phức tạp', 3),
        ('841434', 'Phân tích và chuẩn hoá lược đồ CSDL đến 3NF', 4),
        ('841434', 'Đánh giá hiệu năng truy vấn và đề xuất tối ưu index', 5),
        ('841470', 'Trình bày nguyên lý hoạt động của mạng nơ-ron', 1),
        ('841470', 'Giải thích thuật toán gradient descent', 2),
        ('841470', 'Huấn luyện mô hình phân loại đơn giản bằng scikit-learn', 3),
        ('841470', 'Phân tích kết quả confusion matrix và đề xuất cải tiến', 4),
        ('841470', 'Đánh giá và lựa chọn mô hình ML phù hợp cho bài toán thực', 5),
        ('841470', 'Thiết kế pipeline ML end-to-end cho bài toán phân loại văn bản', 6),
        ('841020', 'Có ý thức bảo vệ quyền sở hữu trí tuệ phần mềm', 0),
    ]
    df = pd.DataFrame(SYNTHETIC_CLOS, columns=['course_code', 'text', 'bloom_level'])

# Đảm bảo cột cần thiết
assert 'text' in df.columns and 'bloom_level' in df.columns
if 'course_code' not in df.columns:
    df['course_code'] = 'unknown'

df['bloom_level'] = df['bloom_level'].astype(int)
print(f'Tổng CLO: {len(df)}')
print(df['bloom_level'].value_counts().sort_index())

## 2. B0 – Rule-based Baseline

In [ ]:
# Danh sách động từ đặc trưng từng lớp Bloom (tóm gọn từ labeling guideline)
BLOOM_VERBS = {
    0: ['có ý thức', 'tôn trọng', 'nhiệt tình', 'chủ động', 'có trách nhiệm',
        'yêu thích', 'trân trọng', 'trung thực', 'liêm chính', 'hợp tác'],
    1: ['trình bày', 'liệt kê', 'nêu', 'định nghĩa', 'nhắc lại', 'ghi nhớ',
        'nhận ra', 'phát biểu', 'chỉ ra', 'đặt tên', 'kể tên', 'lặp lại'],
    2: ['giải thích', 'mô tả', 'diễn giải', 'phân loại', 'tóm tắt',
        'minh họa', 'suy luận', 'dự đoán', 'lý giải', 'ước lượng'],
    3: ['áp dụng', 'thực hiện', 'sử dụng', 'lập trình', 'viết', 'cài đặt',
        'triển khai', 'xây dựng', 'giải', 'tính toán', 'vận dụng', 'thực hành',
        'cấu hình', 'huấn luyện'],
    4: ['phân tích', 'so sánh', 'kiểm tra', 'lập luận', 'đối chiếu', 'phân rã',
        'xác định', 'biện luận', 'truy vết', 'khái quát hóa', 'chuẩn hoá'],
    5: ['đánh giá', 'nhận định', 'phê bình', 'bảo vệ', 'lựa chọn', 'kiểm định',
        'phán xét', 'thẩm định', 'xác nhận', 'phản bác', 'kết luận', 'xếp hạng'],
    6: ['thiết kế', 'phát triển', 'sáng tạo', 'đề xuất', 'lập kế hoạch',
        'soạn thảo', 'tổng hợp', 'tạo ra', 'tích hợp', 'khởi tạo', 'cải tiến',
        'mô hình hóa', 'hiện thực hoá'],
}


def b0_rule_predict(text: str) -> int:
    """Phân loại Bloom bằng rule-based: tìm lớp cao nhất có động từ khớp."""
    text_lower = text.lower()
    matched_level = -1
    # Ưu tiên lớp 0 trước (thái độ)
    for level in [0, 6, 5, 4, 3, 2, 1]:  # ưu tiên cao trước
        for verb in BLOOM_VERBS[level]:
            if verb in text_lower:
                if matched_level < level and level > 0:
                    matched_level = level
                elif level == 0:
                    # Lớp 0 ưu tiên nếu tìm thấy
                    return 0
    return matched_level if matched_level >= 0 else 1  # default: Lớp 1


df['b0_pred'] = df['text'].apply(b0_rule_predict)

# Tính macro F1
from sklearn.metrics import f1_score
b0_macro_f1 = f1_score(df['bloom_level'], df['b0_pred'], average='macro', zero_division=0)
print(f'B0 Macro-F1 (train set): {b0_macro_f1:.4f}')
print()
print(classification_report(df['bloom_level'], df['b0_pred'],
                             target_names=[f'L{i}' for i in range(7)],
                             zero_division=0))

## 3. B1 – TF-IDF + Logistic Regression (5-fold CV theo môn)

In [ ]:
X = df['text'].values
y = df['bloom_level'].values
groups = df['course_code'].values

# Pipeline TF-IDF word + char
tfidf_word = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, analyzer='word')
tfidf_char = TfidfVectorizer(ngram_range=(2, 4), max_features=3000, analyzer='char_wb')

def build_features(texts):
    feat_word = tfidf_word.fit_transform(texts) if not hasattr(tfidf_word, 'vocabulary_') else tfidf_word.transform(texts)
    feat_char = tfidf_char.fit_transform(texts) if not hasattr(tfidf_char, 'vocabulary_') else tfidf_char.transform(texts)
    return sp.hstack([feat_word, feat_char])

# Fit trên toàn bộ để có vocabulary
tfidf_word.fit(X)
tfidf_char.fit(X)

X_feat = sp.hstack([tfidf_word.transform(X), tfidf_char.transform(X)])

# 5-fold CV theo môn
cv = StratifiedGroupKFold(n_splits=min(5, len(np.unique(groups))), shuffle=True, random_state=RANDOM_SEED)
clf = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=RANDOM_SEED)

cv_results = cross_validate(clf, X_feat, y, groups=groups, cv=cv,
                            scoring='f1_macro', return_train_score=True)

b1_cv_mean = cv_results['test_score'].mean()
b1_cv_std = cv_results['test_score'].std()
print(f'B1 5-fold CV Macro-F1: {b1_cv_mean:.4f} ± {b1_cv_std:.4f}')
print(f'Train scores: {cv_results["train_score"]}')
print(f'Val   scores: {cv_results["test_score"]}')

In [ ]:
# Huấn luyện mô hình cuối trên toàn bộ dữ liệu
clf_final = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=RANDOM_SEED)
clf_final.fit(X_feat, y)

# Lưu model
model_bundle = {
    'tfidf_word': tfidf_word,
    'tfidf_char': tfidf_char,
    'clf': clf_final,
    'bloom_names': BLOOM_NAMES,
    'model_type': 'b1_tfidf_logreg',
    'cv_macro_f1_mean': b1_cv_mean,
    'cv_macro_f1_std': b1_cv_std,
}

with open(MODELS_DIR / 'bloom_model.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)

print(f'Đã lưu mô hình B1 → models/bloom_model.pkl')

# Confusion matrix
y_pred_full = clf_final.predict(X_feat)
from src.syllabus_core.visualization.visualize import plot_confusion_matrix
plot_confusion_matrix(
    y, y_pred_full,
    labels=[f'L{i}' for i in range(7)],
    title='Confusion Matrix B1 (train set)',
    save_path=str(FIG_DIR / 'm2_b1_confusion_matrix.png'),
)
print('Đã lưu: reports/figures/m2_b1_confusion_matrix.png')

## 4. Ghi kết quả vào metrics.json

In [ ]:
with open(METRICS_PATH, encoding='utf-8') as f:
    metrics = json.load(f)

metrics['m2_bloom']['experiments']['b0_rule_baseline']['cv_macro_f1_mean'] = round(float(b0_macro_f1), 4)
metrics['m2_bloom']['experiments']['b0_rule_baseline']['status'] = 'completed'

metrics['m2_bloom']['experiments']['b1_tfidf_logreg']['cv_macro_f1_mean'] = round(float(b1_cv_mean), 4)
metrics['m2_bloom']['experiments']['b1_tfidf_logreg']['cv_macro_f1_std'] = round(float(b1_cv_std), 4)
metrics['m2_bloom']['experiments']['b1_tfidf_logreg']['best_C'] = 1.0
metrics['m2_bloom']['experiments']['b1_tfidf_logreg']['status'] = 'completed'

with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f'Đã ghi metrics:')
print(f'  B0 Macro-F1 = {b0_macro_f1:.4f}')
print(f'  B1 CV Mean  = {b1_cv_mean:.4f} ± {b1_cv_std:.4f}')
target = metrics['m2_bloom']['target_macro_f1']
status = '✅ ĐẠT' if b1_cv_mean >= target else '❌ CHƯA ĐẠT'
print(f'  Target {target}: {status}')